<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Pipelines - Solution

---

# Pipeline Implementation

## DataFrame

In [1]:
import pandas as pd

# abalone is a marine sea snail 

abalone_colnames = [
    'sex', 'length', 'diameter', 'height', 'whole_weight',
    'shucked_weight', 'viscera_weight', 'shell_weight', 'rings'
]
ab = pd.read_csv('../data/abalone.data', names=abalone_colnames)
ab.head()

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [2]:
ab.shape

(4177, 9)

In [3]:
ab.dtypes

sex                object
length            float64
diameter          float64
height            float64
whole_weight      float64
shucked_weight    float64
viscera_weight    float64
shell_weight      float64
rings               int64
dtype: object

In [4]:
X = ab.drop(columns=['rings'])
y = ab['rings']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.33, random_state = 10)

## Simple Pipeline

In [6]:
# Libraries
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

```python
#syntak
step1 = ("imputer", SimpleImputer() )
step2 = ('scaling', StandardScaler() )

step_list = [step1,step2 ]

mypipe = Pipeline(step_list)

mypipe.fit_transform( X_train[["age","bmi"]]  )
```

In [7]:
# Prepare the steps to be applied in your preprocessing pipeline
step1 = ("imputer", SimpleImputer() )
step2 = ('scaler', StandardScaler() )

step_list = [step1,step2 ]

# Initialize a pipeline with your created steps
num_transformer = Pipeline(step_list)
num_transformer

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

In [8]:
# Transform length feature
num_transformer.fit_transform( X_train[["length"]]  )

array([[ 0.58216047],
       [ 1.76899281],
       [-3.06311029],
       ...,
       [-0.51989813],
       [ 0.49738673],
       [-0.18080318]])

In [9]:
# Access steps
num_transformer[1] #by index

StandardScaler()

In [10]:
# Access steps
num_transformer['scaler'] #by step name

StandardScaler()

In [11]:
#applying a single step to my feature
num_transformer[0].fit_transform(X[["height","whole_weight"]])

array([[0.095 , 0.514 ],
       [0.09  , 0.2255],
       [0.135 , 0.677 ],
       ...,
       [0.205 , 1.176 ],
       [0.15  , 1.0945],
       [0.195 , 1.9485]])

## Column Transformer

In [12]:
# Libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

```python
#syntax 
branch1 = ("num_tr",num_transformer,["age","bmi"] )
branch2 = ("cat_tr",cat_transformer, ["smoker","region"])

list_branches = [branch1,branch2]

prepro = ColumnTransformer(list_branches)
```

In [13]:
# Impute then Scale for numerical variables: 
num_transformer = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())])

# Encode categorical variables
cat_transformer = OneHotEncoder(
                handle_unknown='ignore', 
                sparse_output=False,
                drop="first")

# Select the features for each transformation/branch
num_features = X_train.select_dtypes(include="number").columns
cat_features = X_train.select_dtypes(include="object").columns

# Paralellize "num_transformer" and "One hot encoder"
preprocessor = ColumnTransformer( [
    ('num_tr', num_transformer, num_features),
    ('cat_tr', cat_transformer, cat_features ) ] )

preprocessor

# a branch can be either a single transformer or a pipeline


ColumnTransformer(transformers=[('num_tr',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight', 'shell_weight'],
      dtype='object')),
                                ('cat_tr',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 Index(['sex'], dtype='object'))])

In [14]:
# Transform your training set
X_train_transformed = preprocessor.fit_transform(X_train)

X_train_transformed

array([[ 0.58216047,  0.62053938,  0.34523682, ...,  0.1705245 ,
         0.        ,  1.        ],
       [ 1.76899281,  1.95763432,  1.86112818, ...,  2.58196828,
         0.        ,  0.        ],
       [-3.06311029, -2.97933159, -2.56993888, ..., -1.70104381,
         1.        ,  0.        ],
       ...,
       [-0.51989813, -0.56227536, -0.5876194 , ..., -0.83004471,
         0.        ,  0.        ],
       [ 0.49738673,  0.62053938,  0.57845088, ...,  0.38647469,
         0.        ,  1.        ],
       [-0.18080318, -0.15086154, -0.82083346, ..., -0.50611943,
         1.        ,  0.        ]])

## Custom Transformer (Stateless Transformation)

In [15]:
# Libraries
from sklearn.preprocessing import FunctionTransformer
import numpy as np


In [16]:
from sklearn.preprocessing import FunctionTransformer

# Create a transformer that logs the shell_weight feature
def log_fc (series:  pd.Series) -> pd.Series:
    return np.log(series)

log_tr = FunctionTransformer(log_fc)

# Create a new branch to log this feature
num_features = X_train.select_dtypes(include="number").columns.drop("shell_weight")
cat_features = X_train.select_dtypes(include="object").columns
log_features = ["shell_weight"]

preprocessor = ColumnTransformer( [
    ('num_tr', num_transformer, num_features),
    ('cat_tr', cat_transformer, cat_features ),
    ('log_tr',log_tr, log_features)] )

preprocessor

ColumnTransformer(transformers=[('num_tr',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                ('cat_tr',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 Index(['sex'], dtype='object')),
                                ('log_tr',
                                 FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                 ['shell_weight'])])

In [17]:
# Transform your training set
X_train_transformed = preprocessor.set_output(transform="pandas").fit_transform(X_train)

X_train_transformed.head()

,num_tr__length,num_tr__diameter,num_tr__height,num_tr__whole_weight,num_tr__shucked_weight,num_tr__viscera_weight,cat_tr__sex_I,cat_tr__sex_M,log_tr__shell_weight
2916,0.582160,0.620539,0.345237,0.741990,0.576933,1.873071,0.0,1.0,-1.328025
1051,1.768993,1.957634,1.861128,3.499892,3.456076,2.363570,0.0,0.0,-0.510826
306,-3.063110,-2.979332,-2.569939,-1.662387,-1.599816,-1.624602,1.0,0.0,-5.298317
828,-0.986154,-0.870836,-0.937440,-0.903378,-0.697699,-1.069925,1.0,0.0,-2.244316
3317,-2.215373,-2.156504,-1.753690,-1.535037,-1.478636,-1.574176,0.0,1.0,-3.506558


## Summary with make_*** shortcuts 

```python
#this
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

#is equivalent to this
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
```

```python
# This
Pipeline([
    ('my_name_for_imputer', SimpleImputer()),
    ('my_name_for_scaler', StandardScaler())
])

# Equivalent to this
make_pipeline( SimpleImputer(), StandardScaler() )
```

In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

num_transformer = make_pipeline(SimpleImputer(), StandardScaler())
cat_transformer = OneHotEncoder()
log_transformer = FunctionTransformer(log_fc)

preproc_basic = make_column_transformer( (num_transformer, num_features), #pipeline
                                         (cat_transformer, cat_features), #transfomer
                                         (log_transformer, log_features),) #custom transformer
preproc_basic

ColumnTransformer(transformers=[('pipeline',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer()),
                                                 ('standardscaler',
                                                  StandardScaler())]),
                                 Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                ('onehotencoder', OneHotEncoder(),
                                 Index(['sex'], dtype='object')),
                                ('functiontransformer',
                                 FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                 ['shell_weight'])])

## Including Model in a Pipeline

In [19]:
# Add estimator
from sklearn.linear_model import Ridge

full_pipeline = make_pipeline(preproc_basic, Ridge())
full_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(),
                                                  Index(['sex'], dtype='object')),
                                                 ('functiontransformer',
                                                  FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                                  ['shell_weight'])])),
                ('ridge', Ridge())])

In [20]:
# Train pipeline

full_pipeline.fit(X_train,y_train)

# .fit() does both, fit & transform data then train model

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(),
                                                  Index(['sex'], dtype='object')),
                                                 ('functiontransformer',
                                                  FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                                  ['shell_weight'])])),
                ('ridge', Ridge())])

In [21]:
# Make predictions

full_pipeline.predict(X_test)

# .predict() will transform your data first before predicting

array([ 8.24518219, 10.50465703,  9.88362871, ...,  9.36153899,
        9.98776983, 10.2254665 ])

In [22]:
# Score model
full_pipeline.score(X_test,y_test)

# .score() will transform your data first too

0.5454349761940098

## Cross Validate a Pipeline

In [23]:
from sklearn.model_selection import cross_val_score

# Cross validate pipeline
cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring='r2').mean()

0.5425313592838956

## GridSearch a Pipeline

In [24]:
full_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(),
                                                  Index(['sex'], dtype='object')),
                                                 ('functiontransformer',
                                                  FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                                  ['shell_weight'])])),
                ('ridge', Ridge())])

In [25]:
#### Get all pipe components parameters (to find hyper params names)
full_pipeline.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('pipeline',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer()),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
          'viscera_weight'],
         dtype='object')),
                                   ('onehotencoder', OneHotEncoder(),
                                    Index(['sex'], dtype='object')),
                                   ('functiontransformer',
                                    FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                    ['shell_weight'])])),
  ('ridge', Ridge())],
 'verbose': False,
 'columntransformer': ColumnTransformer(transformers=[('p

In [26]:
from sklearn.model_selection import GridSearchCV

# Instanciate grid search
grid_search = GridSearchCV(
    full_pipeline, 
    param_grid={
        # Access any component of the pipeline, as far back as you want
        'columntransformer__pipeline__simpleimputer__strategy': ['mean', 'median'],
        'ridge__alpha': [0.1, 0.5, 1, 5, 10]},
    cv=5,
    scoring="r2")

grid_search.fit(X_train, y_train)
grid_search.best_params_

{'columntransformer__pipeline__simpleimputer__strategy': 'mean',
 'ridge__alpha': 0.1}

In [27]:
full_pipeline_tuned = grid_search.best_estimator_

In [28]:
full_pipeline_tuned.score(X_test,y_test)

0.5447797230043703

In [29]:
full_pipeline_tuned.predict(X_test[0:1])

array([8.2338286])

## Caching to avoid repeated computations

This becomes important when your pipeline contains expensive preprocessing steps and you are using things like GridSearchCV / cross-validation, because the same transformations may otherwise be recomputed many times.


```python
#Syntax
from tempfile import mkdtemp
from shutil import rmtree

# Create a temp folder
cachedir = mkdtemp()

# Instantiate the pipeline with cache parameter
pipe = Pipeline(steps, memory=cachedir)

# Clear the cache directory after the cross-validation
rmtree(cachedir)
```

## Debugging Your Pipeline

In [30]:
full_pipeline_tuned

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['length', 'diameter', 'height', 'whole_weight', 'shucked_weight',
       'viscera_weight'],
      dtype='object')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(),
                                                  Index(['sex'], dtype='object')),
                                                 ('functiontransformer',
                                                  FunctionTransformer(func=<function log_fc at 0x147794ca0>),
                                                  ['shell_weight'])])),
                ('ridge', Ridge(alpha=0.1))])

In [31]:
# Access component of pipeline with `name_steps`
full_pipeline_tuned.named_steps.keys()

dict_keys(['columntransformer', 'ridge'])

In [32]:
# Check intermediate steps
print("Before preprocessing, X_train.shape = ")
print(X_train.shape)
print("After preprocessing, X_train_preprocessed.shape = ")
full_pipeline_tuned.named_steps["columntransformer"].fit_transform(X_train).shape

Before preprocessing, X_train.shape = 
(2798, 8)
After preprocessing, X_train_preprocessed.shape = 


(2798, 10)

##  Exporting models/pipelines

In [33]:
import pickle

# Export Pipeline as pickle file
with open("pipeline.pkl", "wb") as file:
    pickle.dump(full_pipeline_tuned, file)

# Load Pipeline from pickle file
my_pipeline = pickle.load(open("pipeline.pkl","rb"))

my_pipeline.score(X_test, y_test)

0.5447797230043703